<a href="https://colab.research.google.com/github/AlexandreLouzada/exercicios-analise-dados/blob/master/matplotlib_varejo_GABARITO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Matplotlib do Básico ao Avançado: BI do Varejo — GABARITO

**Contexto:** Time de BI de rede varejista. Base com `data, categoria, estado, valor, quantidade, valor_venda` (120 registros, 2024).

**Roteiro:** Parte 1 (gráficos essenciais) → Parte 2 (OO, subplots, twinx, annotate) → Parte 3 (pivot + heatmap com imshow + savefig).

**Colab:** `Arquivo > Fazer upload do notebook`, executar com `Shift + Enter`. Só `matplotlib`, `numpy`, `pandas`.

---

## 🧱 Configuração do Ambiente e Dados Iniciais

▶️ Execute primeiro.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Gerando dados sintéticos estruturados
np.random.seed(42)
n_registros = 120

datas = pd.date_range(start='2024-01-01', periods=n_registros, freq='D')
categorias = ['Eletronicos', 'Livros', 'Roupas', 'Automotivo']
estados = ['SP', 'RJ', 'MG', 'RS', 'BA']

dados = {
    'data': datas,
    'categoria': np.random.choice(categorias, size=n_registros),
    'estado': np.random.choice(estados, size=n_registros),
    'valor': np.random.uniform(50, 4500, size=n_registros).round(2),
    'quantidade': np.random.randint(1, 15, size=n_registros),
}

df = pd.DataFrame(dados)
df['valor_venda'] = (df['valor'] * np.random.uniform(0.9, 1.1, size=n_registros)).round(2)

print(df.head().to_string())
print()
df.info()

# Série mensal de apoio (reutilizada nas tarefas 1, 4, 5, 6)
df_mensal = df.set_index('data').resample('ME').agg(faturamento=('valor_venda', 'sum'), qtd_total=('quantidade', 'sum'), qtd_media=('quantidade', 'mean'))
print()
print(df_mensal.round(2).to_string())

## 🟢 Parte 1 — Gráficos Essenciais e Personalização

### Tarefa 1: Linha do faturamento mensal
`resample('ME')`, marcador `'o'`, linha `'--'`, cor customizada, `grid`, rótulos X/Y.

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(df_mensal.index, df_mensal['faturamento'],
         marker='o', linestyle='--', color='#023e8a', linewidth=2)
plt.title('Faturamento Mensal Total — 2024', fontsize=14, fontweight='bold')
plt.xlabel('Mês', fontsize=12)
plt.ylabel('Faturamento (R$)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Tarefa 2: Barras — total de vendas por categoria (ordenado)
Ordenado do menor para o maior, `rotation=45` no eixo X.

In [ ]:
vendas_cat = df.groupby('categoria')['valor_venda'].sum().sort_values()  # menor -> maior
print(vendas_cat.round(2).to_string())

plt.figure(figsize=(9, 5))
plt.bar(vendas_cat.index, vendas_cat.values, color='#219ebc', edgecolor='black')
plt.title('Total de Vendas por Categoria (ordenado)', fontsize=14, fontweight='bold')
plt.xlabel('Categoria', fontsize=12)
plt.ylabel('Total vendido (R$)', fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Tarefa 3: Histograma de `valor` (10 bins, bordas pretas)

In [ ]:
plt.figure(figsize=(9, 5))
plt.hist(df['valor'], bins=10, color='#ffb703', edgecolor='black')
plt.title('Distribuição dos Valores de Venda', fontsize=14, fontweight='bold')
plt.xlabel('Valor (R$)', fontsize=12)
plt.ylabel('Frequência', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 🟡 Parte 2 — API Orientada a Objetos e Eixos Múltiplos

### Tarefa 4: Subplots 2×1 com `sharex=True`
ax1: linha do faturamento mensal | ax2: barras da quantidade total no mesmo período.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

# ax1 — linha faturamento
ax1.plot(df_mensal.index, df_mensal['faturamento'], marker='o', linestyle='-', color='#023e8a')
ax1.set_title('Faturamento Mensal (linha)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Faturamento (R$)')
ax1.grid(True, linestyle='--', alpha=0.6)

# ax2 — barras quantidade total
ax2.bar(df_mensal.index, df_mensal['qtd_total'], color='#fb8500', edgecolor='black', width=20)
ax2.set_title('Quantidade Total de Itens por Mês (barras)', fontsize=12, fontweight='bold')
ax2.set_xlabel('Mês')
ax2.set_ylabel('Qtd. total')
ax2.tick_params(axis='x', rotation=45)

fig.suptitle('Painel Mensal: Faturamento × Quantidade', fontsize=14, fontweight='bold')
fig.tight_layout()
plt.show()

### Tarefa 5: Eixo duplo com `twinx()`
Esquerda (barras): valor total/mês | Direita (linha + marcadores): quantidade média por venda.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.bar(df_mensal.index, df_mensal['faturamento'], color='#8ecae6', edgecolor='black', label='Faturamento', width=20)
ax.set_xlabel('Mês')
ax.set_ylabel('Faturamento (R$)', color='#023e8a')
ax.tick_params(axis='x', rotation=45)

ax2 = ax.twinx()
ax2.plot(df_mensal.index, df_mensal['qtd_media'], color='red', marker='s', linewidth=2, label='Qtd. média/venda')
ax2.set_ylabel('Qtd. média por venda', color='red')

plt.title('Faturamento (barras) × Qtd. Média por Venda (linha)', fontsize=13, fontweight='bold')
fig.tight_layout()
plt.show()

### Tarefa 6: Anotação do pico com `ax.annotate()`

In [ ]:
mes_pico = df_mensal['faturamento'].idxmax()
valor_pico = df_mensal['faturamento'].max()
print(f"Pico: {mes_pico.strftime('%Y-%m')} — R$ {valor_pico:,.2f}")

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(df_mensal.index, df_mensal['faturamento'], marker='o', linestyle='--', color='#023e8a')
ax.set_title('Faturamento Mensal com Pico Anotado', fontsize=14, fontweight='bold')
ax.set_xlabel('Mês')
ax.set_ylabel('Faturamento (R$)')
ax.grid(True, linestyle='--', alpha=0.6)
ax.annotate(f'Pico: R$ {valor_pico:,.0f}',
            xy=(mes_pico, valor_pico), xycoords='data',
            xytext=(15, 25), textcoords='offset points',
            arrowprops=dict(arrowstyle='->', color='red'),
            fontsize=10, color='red', fontweight='bold')
fig.autofmt_xdate(rotation=45)
fig.tight_layout()
plt.show()

## 🔴 Parte 3 — Matriz de Intensidade / Heatmap

### Tarefas 7–9: `pivot_table` → `imshow` + `colorbar` + ticks → `savefig` 300 DPI

In [ ]:
# Tarefa 7: pivot categoria (linhas) × estado (colunas), soma de valor_venda
pivot = pd.pivot_table(df, values='valor_venda', index='categoria', columns='estado', aggfunc='sum').round(2)
display(pivot)

# Tarefa 8: heatmap com imshow
plt.figure(figsize=(9, 6))
im = plt.imshow(pivot.values, aspect='auto')
plt.colorbar(im, label='Soma valor_venda (R$)')
plt.xticks(ticks=range(len(pivot.columns)), labels=pivot.columns)
plt.yticks(ticks=range(len(pivot.index)), labels=pivot.index)
plt.xlabel('Estado')
plt.ylabel('Categoria')
plt.title('Heatmap: Vendas por Categoria × Estado', fontsize=14, fontweight='bold')

# Rótulos de valor dentro das células (opcional, ajuda a leitura)
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        plt.text(j, i, f"{pivot.values[i, j]:,.0f}", ha='center', va='center', fontsize=8)

plt.tight_layout()

# Tarefa 9: salvar em alta resolução
plt.savefig('heatmap_vendas.png', dpi=300, bbox_inches='tight')
plt.show()
print("Figura salva como 'heatmap_vendas.png' (300 DPI, bbox_inches='tight').")

### 🚀 Desafio extra

1. Troque o `cmap`: `plt.imshow(..., cmap='viridis')` e compare.
2. Normalize por linha para ver o perfil regional de cada categoria.
3. Monte um dashboard 2×2 reunindo linha + barras + hist + heatmap.